|                |   |
:----------------|---|
| **Nombre**     | Valeria Guzmán Galván  |
| **Fecha**      | 04/05/2026  |
| **Expediente** | 756902  |

# Bootstraping

## Actividad

1. Tomar el dataset "Motor Trend Road"

2. Realizar regresión lineal sobre mpg = b0 + b1(hp) + b2(qsec)

   * Calcular intervalos de confianza 95% de los betas

3. Crear 1000 muestras de bootstrap con m observaciones con reemplazo

   * Realizar la regresión mpg = b0 + b1(hp) + b2(qsec)

   * Calcular Bx y sigma_b, abrir intervalos de confianza

4. Comparar resultados entre inciso 2 y 3


In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import seaborn as sns

In [4]:
from google.colab import files
uploaded = files.upload()

Saving Motor Trend Car Road Tests.xlsx to Motor Trend Car Road Tests.xlsx


In [5]:
df = pd.read_excel("Motor Trend Car Road Tests.xlsx")
df.head()

,model,mpg,cyl,disp,hp,drat,wt,qsec,vs,am,gear,carb
0,Mazda RX4,21.0,6,160.0,110,3.90,2.620,16.46,0,1,4,4
1,Mazda RX4 Wag,21.0,6,160.0,110,3.90,2.875,17.02,0,1,4,4
2,Datsun 710,22.8,4,108.0,93,3.85,2.320,18.61,1,1,4,1
3,Hornet 4 Drive,21.4,6,258.0,110,3.08,3.215,19.44,1,0,3,1
4,Hornet Sportabout,18.7,8,360.0,175,3.15,3.440,17.02,0,0,3,2


## 2. Regresión Lineal

In [6]:
X = df[['hp', 'qsec']]
X = sm.add_constant(X)  # beta0

y = df['mpg']

In [7]:
modelo = sm.OLS(y, X).fit()

In [8]:
print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                    mpg   R-squared:                       0.637
Model:                            OLS   Adj. R-squared:                  0.612
Method:                 Least Squares   F-statistic:                     25.43
Date:                Mon, 04 May 2026   Prob (F-statistic):           4.18e-07
Time:                        22:39:56   Log-Likelihood:                -86.170
No. Observations:                  32   AIC:                             178.3
Df Residuals:                      29   BIC:                             182.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         48.3237     11.103      4.352      0.0

### 2.1 IC 95% de los betas

In [9]:
ic = modelo.conf_int(alpha=0.05)
ic.columns = ['Límite inferior', 'Límite superior']
print(ic)

       Límite inferior  Límite superior
const        25.614894        71.032516
hp           -0.113089        -0.056097
qsec         -1.979929         0.206770


## 3. 1000 muestras con Bootstrap

In [10]:
n = len(df)
betas = []

In [11]:
for i in range(1000):
    # muestra con reemplazo
    muestra = df.sample(n=n, replace=True)

    X_b = muestra[['hp', 'qsec']]
    X_b = sm.add_constant(X_b)
    y_b = muestra['mpg']

    modelo_b = sm.OLS(y_b, X_b).fit()

    betas.append(modelo_b.params.values)

betas = np.array(betas)

### 3.2 IC 95% bootstrap

In [12]:
medias = betas.mean(axis=0)
stds = betas.std(axis=0)

print("Medias:", medias)
print("Desviaciones:", stds)

Medias: [50.40191181 -0.08803004 -0.97978593]
Desviaciones: [10.93939859  0.01676181  0.52798711]


In [13]:
ic_df = pd.DataFrame({
    "Límite Inferior": medias - 2*stds,
    "Límite Superior": medias + 2*stds
}, index=["b0", "b1", "b2"])

print(ic_df)

    Límite Inferior  Límite Superior
b0        28.523115        72.280709
b1        -0.121554        -0.054506
b2        -2.035760         0.076188


## 4. Comparación inciso 2 y 3

Al comparar los resultados del método clásico y el bootstrap, se observa que ambos producen intervalos de confianza muy similares para los coeficientes del modelo.

En particular, el coeficiente de hp es significativo en ambos métodos, ya que sus intervalos no incluyen el valor 0. Por otro lado, el coeficiente de qsec no es significativo en ninguno de los dos casos, ya que sus intervalos sí incluyen el 0.

Esto muestra que ambos métodos llegan a las mismas conclusiones, por lo que los resultados del modelo son confiables.

# Aggregating

## Actividad

1. Dividir los datos en train y test (50/50)
2. Generar 1000 modelos de regresión lineal, seleccionando en cada uno 3 variables al azar
3. Entrenar cada modelo con train y predecir sobre test
4. Calcular el R2 de cada modelo
5. Promediar las predicciones de los 1000 modelos para obtener ytest
6. Evaluar el modelo agregado calculando el R2 final

In [14]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

In [15]:
X = df.drop(columns=["mpg", "model"])
y = df["mpg"]

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

In [17]:
columnas = X.columns
resultados = []

In [18]:
for i in range(1000):
    vars_random = np.random.choice(columnas, size=3, replace=False) # 3 variables al azar
    Xi_train = X_train[list(vars_random)]

    modelo = LinearRegression()
    modelo.fit(Xi_train, y_train)

    resultados.append({
        "variables": vars_random,
        "modelo": modelo
    })

In [19]:
predicciones = []

In [20]:
for res in resultados:
    vars_modelo = res["variables"]
    modelo = res["modelo"]

    Xi_test = X_test[list(vars_modelo)]  # datos de test con mismas variables

    y_pred = modelo.predict(Xi_test)
    predicciones.append(y_pred)

In [21]:
predicciones = np.array(predicciones)
predicciones.shape

(1000, 16)

In [22]:
y_pred_promedio = predicciones.mean(axis=0)
y_pred_promedio

array([20.34789125, 10.07598234, 14.41362299, 27.11245749, 23.51680029,
       20.15214775, 13.69570563, 27.49129334, 15.29550405, 21.83521527,
       15.48299439, 10.31637154, 19.69724817, 15.25762736, 14.69773794,
       13.57535044])

In [23]:
r2_final = r2_score(y_test, y_pred_promedio)
print("R2 final (bagging):", r2_final)

R2 final (bagging): 0.7819336014537466


El valor de R2 indica que el modelo explica aproximadamente el 78.8% de la variabilidad de la variable mpg en el conjunto de prueba, lo cual representa un buen ajuste.

Esto muestra que el uso de aggregating permite mejorar la estabilidad y el desempeño del modelo, ya que al promediar múltiples modelos se reduce la variabilidad de las predicciones.

# Random Forest

In [70]:
from sklearn.model_selection import KFold, RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor

In [71]:
X = df.drop(columns=["mpg", "model"])
y = df["mpg"]

In [72]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

rf = RandomForestRegressor(random_state=42)

In [73]:
param_dist = {
    'n_estimators': [200, 300, 500, 800],
    'max_depth': [None, 3, 4, 5, 6, 8],
    'max_leaf_nodes': [None, 10, 15, 20, 30],
}

In [74]:
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=50,
    cv=kfold,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

In [75]:
random_search.fit(X, y)

RandomizedSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
                   estimator=RandomForestRegressor(random_state=42), n_iter=50,
                   n_jobs=-1,
                   param_distributions={'max_depth': [None, 3, 4, 5, 6, 8],
                                        'max_leaf_nodes': [None, 10, 15, 20,
                                                           30],
                                        'n_estimators': [200, 300, 500, 800]},
                   random_state=42, scoring='r2')

In [76]:
print("Mejores parámetros:", random_search.best_params_)
print("Mejor R2:", random_search.best_score_)

Mejores parámetros: {'n_estimators': 500, 'max_leaf_nodes': None, 'max_depth': 4}
Mejor R2: 0.5116072603953695


Con los hiperparámteros que encontramos hacemos un nuevo forest, entrenamos el modelo con todos los datos y obtenemos r2

In [79]:
rf_final = RandomForestRegressor(n_estimators=500, max_depth=4, max_leaf_nodes=None, random_state=42)

rf_final.fit(X, y)

RandomForestRegressor(max_depth=4, n_estimators=500, random_state=42)

In [80]:
y_pred = rf_final.predict(X)

In [81]:
r2_final = r2_score(y, y_pred)
print("R2 final (modelo con todos los datos):", r2_final)

R2 final (modelo con todos los datos): 0.9723145983864327


# Gradient Boosting Regression

In [82]:
from sklearn.ensemble import GradientBoostingRegressor

In [83]:
X = df.drop(columns=['mpg', 'model'])
y = df['mpg']

In [84]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)

gbr = GradientBoostingRegressor(random_state=42)

In [96]:
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4, 5],
    'max_leaf_nodes': [10, 15, 20, 30]
}

In [97]:
random_search_gb = RandomizedSearchCV(
    estimator=gbr,
    param_distributions=param_dist,
    n_iter=30,
    cv=kfold,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

In [98]:
random_search_gb.fit(X, y)

RandomizedSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
                   estimator=GradientBoostingRegressor(random_state=42),
                   n_iter=30, n_jobs=-1,
                   param_distributions={'max_depth': [2, 3, 4, 5],
                                        'max_leaf_nodes': [10, 15, 20, 30],
                                        'n_estimators': [100, 200, 300]},
                   random_state=42, scoring='r2')

In [99]:
print("Mejores parámetros:", random_search_gb.best_params_)
print("Mejor R2:", random_search_gb.best_score_)

Mejores parámetros: {'n_estimators': 100, 'max_leaf_nodes': 15, 'max_depth': 2}
Mejor R2: 0.4254818274225093


In [103]:
gbr_final = GradientBoostingRegressor(n_estimators=100, max_leaf_nodes=15, max_depth=2, random_state=42)

gbr_final.fit(X, y)

GradientBoostingRegressor(max_depth=2, max_leaf_nodes=15, random_state=42)

In [104]:
y_pred = gbr_final.predict(X)

In [105]:
r2_final = r2_score(y, y_pred)
print("R2 final:", r2_final)

R2 final: 0.9977739884014936
